# Silver Layer Transformation

## Read Bronze Data

This step loads the raw Raider.io Mythic+ run data from the Bronze table.

The Bronze layer contains the original API data with minimal processing.  
Data is stored in its raw nested JSON structure along with ingestion metadata.

In the Silver layer we begin transforming this raw data by:

- Reading the Bronze Delta table
- Inspecting the schema
- Preparing the dataset for normalization and flattening
- Creating structured tables for dungeon runs and player information

The data is loaded from the Bronze table:

workspace.bronze_mythic_plus_runs.mythic_runs_bronze

This table contains the raw Mythic+ run records ingested from the Raider.io API.

Next steps in this notebook will focus on:
- Exploring the schema
- Flattening nested structures
- Exploding player roster arrays
- Writing cleaned data into Silver tables

In [0]:
df_bronze = spark.table("workspace.bronze_mythic_plus_runs.mythic_runs_bronze")

display(df_bronze)

In [0]:
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, when, date_format, to_timestamp

dungeons_df = df_bronze.select(
    col("keystone_run_id"),
    col("keystone_team_id"),
    col("season"),
    col("score"),
    col("clear_time_ms"),
    col("keystone_time_ms"),
    col("completed_at"),
    
    # dungeon struct fields
    col("dungeon.id").alias("dungeon_id"),
    col("dungeon.name").alias("dungeon_name"),
    col("dungeon.slug").alias("dungeon_slug"),
    col("dungeon.expansion_id").alias("expansion_id"),
    col("dungeon.keystone_timer_ms").alias("timer_ms")
)

# Clean Datetime format
dungeons_df = (
    dungeons_df
    .withColumn("completed_at", date_format(to_timestamp(col("completed_at")), "yyyy-MM-dd HH:mm:ss"))
)

# Clean Season naming format
dungeons_df = dungeons_df.withColumn(
    "season",
    when(col("season") == "season-tww-1", "The War Within Season 1")
    .when(col("season") == "season-df-1", "Dragonflight Season 1")
    .when(col("season") == "season-bfa-4", "Battle for Azeroth Season 4")
    .otherwise(col("season"))
)

dungeons_df.printSchema()

display(dungeons_df)

In [0]:
dungeons_df.write.mode("overwrite").saveAsTable(
    "workspace.silver_mythic_plus.dungeons"
)

In [0]:
from pyspark.sql.functions import explode, col, date_format, to_timestamp

boss_encounters_df = (
    df_bronze
    .withColumn("encounter", explode("logged_details.encounters"))
    .select(
        col("keystone_run_id"),

        col("encounter.id").alias("encounter_id"),
        col("encounter.duration_ms").alias("fight_duration_ms"),
        col("encounter.is_success").alias("boss_killed"),

        col("encounter.pull_started_at"),
        col("encounter.pull_ended_at"),

        col("encounter.boss.name").alias("boss_name"),
        col("encounter.boss.encounterId").alias("boss_id"),
        col("encounter.boss.slug").alias("boss_slug")
    )
)

# Clean Datetime format
boss_encounters_df = (
    boss_encounters_df
    .withColumn("pull_started_at", date_format(to_timestamp(col("pull_started_at")), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("pull_ended_at", date_format(to_timestamp(col("pull_ended_at")), "yyyy-MM-dd HH:mm:ss"))
)

boss_encounters_df.printSchema()

display(boss_encounters_df)

In [0]:
boss_encounters_df.write.mode("overwrite").saveAsTable(
    "workspace.silver_mythic_plus.boss_encounters"
)

In [0]:
from pyspark.sql.functions import explode, col, regexp_replace, when

players_df = (
    df_bronze
    .withColumn("player", explode("roster"))
    .select(
        col("keystone_run_id"),

        col("player.character.name").alias("character_name"),
        col("player.character.level").alias("level"),

        col("player.character.class.name").alias("class"),
        col("player.character.spec.name").alias("spec"),
        col("player.character.spec.role").alias("role"),

        col("player.character.realm.name").alias("realm"),
        col("player.character.region.name").alias("region")
    )
)

players_df = players_df.withColumn(
    "character_name",
    regexp_replace(col("character_name"), "-.*$", "")
)

players_df = players_df.withColumn(
    "role",
    when(col("role") == "healer", "Healer")
    .when(col("role") == "tank", "Tank")
    .when(col("role") == "dps", "DPS")
    .otherwise(col("role"))
)

players_df.printSchema()

display(players_df)

In [0]:
players_df.write.mode("overwrite").saveAsTable(
    "workspace.silver_mythic_plus.players"
)

In [0]:
display(spark.table("workspace.silver_mythic_plus.players"))
display(spark.table("workspace.silver_mythic_plus.boss_encounters"))
display(spark.table("workspace.silver_mythic_plus.dungeons"))